# DAY - 6

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## Load All Four Datasets

In [ ]:
# Load datasets
interactions = pd.read_csv("../data/raw/llm_system_interactions.csv")
sessions = pd.read_csv("../data/raw/llm_system_sessions_summary.csv")
users = pd.read_csv("../data/raw/llm_system_users_summary.csv")
prompts = pd.read_csv("../data/raw/llm_system_prompts_lookup.csv")

## Verify That All Files Loaded Correctly

In [ ]:
print("Interactions:", interactions.shape)
print("Sessions:", sessions.shape)
print("Users:", users.shape)
print("Prompts:", prompts.shape)

In [ ]:
# Function to trim whitespace from all string columns
def trim_whitespace(df):
    string_cols = df.select_dtypes(include='object').columns
    df[string_cols] = df[string_cols].apply(lambda col: col.str.strip())
    return df

# Apply to all datasets
interactions = trim_whitespace(interactions)
sessions = trim_whitespace(sessions)
users = trim_whitespace(users)
prompts = trim_whitespace(prompts)

print("Whitespace trimmed successfully.")

In [ ]:
# Check if any string column still has leading/trailing spaces
def check_whitespace(df, name):
    string_cols = df.select_dtypes(include='object').columns

    issues = []

    for col in string_cols:
        if df[col].dropna().str.match(r'^\s|\s$').any():
            issues.append(col)

    if issues:
        print(f"{name}: Whitespace found in {issues}")
    else:
        print(f"{name}: No leading/trailing whitespace found.")

check_whitespace(interactions, "Interactions")
check_whitespace(sessions, "Sessions")
check_whitespace(users, "Users")
check_whitespace(prompts, "Prompts")

In [ ]:
# Display data types for each dataset

print("===== Interactions =====")
print(interactions.dtypes)

print("\n===== Sessions =====")
print(sessions.dtypes)

print("\n===== Users =====")
print(users.dtypes)

print("\n===== Prompts =====")
print(prompts.dtypes)

## Validate Missing Values

In [ ]:
# Missing values after cleaning

datasets = {
    "Interactions": interactions,
    "Sessions": sessions,
    "Users": users,
    "Prompts": prompts
}

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)

## Save the Cleaned Datasets

In [ ]:
# Save cleaned datasets

interactions.to_csv("../data/cleaned/llm_system_interactions_cleaned.csv", index=False)
sessions.to_csv("../data/cleaned/llm_system_sessions_summary_cleaned.csv", index=False)
users.to_csv("../data/cleaned/llm_system_users_summary_cleaned.csv", index=False)
prompts.to_csv("../data/cleaned/llm_system_prompts_lookup_cleaned.csv", index=False)

print("All cleaned datasets saved successfully.")

## Validate Row Counts

In [ ]:
# Compare original and cleaned row counts

datasets = [
    ("Interactions", interactions),
    ("Sessions", sessions),
    ("Users", users),
    ("Prompts", prompts)
]

print("Row Count Validation")
print("-" * 40)

for name, df in datasets:
    print(f"{name:<15} {len(df)} rows")

# Day 6 - Data Cleaning Summary

## Cleaning Activities Performed

- Loaded all four datasets.
- Trimmed leading and trailing whitespace from all text columns.
- Verified that no whitespace issues remained.
- Validated data types across all datasets.
- Confirmed expected missing values without modifying them.
- Preserved all original records (no rows removed).
- Saved cleaned datasets in the `data/cleaned/` folder for Exploratory Data Analysis (EDA).

### Result

The datasets are clean, consistent, and ready for EDA.

# Day 11 - Cleaning Validation

## 1. Load Cleaned Datasets

Load all cleaned CSV files from the cleaned folder for validation.

In [ ]:
# Load cleaned datasets

interactions_clean = pd.read_csv("../data/cleaned/llm_system_interactions_cleaned.csv")

sessions_clean = pd.read_csv("../data/cleaned/llm_system_sessions_summary_cleaned.csv")

users_clean = pd.read_csv("../data/cleaned/llm_system_users_summary_cleaned.csv")

prompts_clean = pd.read_csv("../data/cleaned/llm_system_prompts_lookup_cleaned.csv")

## 2. Validate Row Counts

Verify that each cleaned dataset contains the expected number of records after the cleaning process.

In [ ]:
print("Interactions :", len(interactions_clean))
print("Sessions     :", len(sessions_clean))
print("Users        :", len(users_clean))
print("Prompts      :", len(prompts_clean))

## 3. Check Missing Values

Ensure there are no unexpected missing values remaining in the cleaned datasets.

In [ ]:
print("Interactions")
print(interactions_clean.isnull().sum())

print("\nSessions")
print(sessions_clean.isnull().sum())

print("\nUsers")
print(users_clean.isnull().sum())

print("\nPrompts")
print(prompts_clean.isnull().sum())

## 4. Investigate Remaining Missing Values

Determine whether the detected missing values are stored as actual NaN values or as empty strings in the cleaned datasets.

In [ ]:
# Columns with reported missing values
columns_to_check = {
    "Interactions": (interactions_clean, "user_feedback_score"),
    "Sessions (CSAT)": (sessions_clean, "csat_mean"),
    "Sessions (RPM)": (sessions_clean, "requests_per_minute"),
    "Users": (users_clean, "avg_csat")
}

for name, (df, col) in columns_to_check.items():
    print(f"\n{name} - {col}")
    print("NaN values :", df[col].isna().sum())
    print("Empty strings :", (df[col] == "").sum() if df[col].dtype == "object" else "Not Applicable")

## 5. Check Duplicate Records

Verify that duplicate records have been removed from all cleaned datasets.

In [ ]:
print("Interactions :", interactions_clean.duplicated().sum())
print("Sessions     :", sessions_clean.duplicated().sum())
print("Users        :", users_clean.duplicated().sum())
print("Prompts      :", prompts_clean.duplicated().sum())

## 6. Verify Data Types

Inspect the data types of all columns to ensure they are appropriate for further analysis.

In [ ]:
print("Interactions")
print(interactions_clean.dtypes)

print("\nSessions")
print(sessions_clean.dtypes)

print("\nUsers")
print(users_clean.dtypes)

print("\nPrompts")
print(prompts_clean.dtypes)

## 7. Convert Date and Time Columns

Convert the timestamp and date columns from text (object) format to datetime format to support time-based analysis, trend identification, and dashboard development.

In [ ]:
# Convert date and time columns

interactions_clean["timestamp_utc"] = pd.to_datetime(
    interactions_clean["timestamp_utc"], utc=True
)

interactions_clean["date_utc"] = pd.to_datetime(
    interactions_clean["date_utc"]
)

sessions_clean["start_timestamp_utc"] = pd.to_datetime(
    sessions_clean["start_timestamp_utc"], utc=True
)

sessions_clean["end_timestamp_utc"] = pd.to_datetime(
    sessions_clean["end_timestamp_utc"], utc=True
)

print("Date and time columns converted successfully.")

## 8. Verify Date and Time Conversion

Verify that the converted columns now use the appropriate datetime data type and are ready for time-based analysis.

In [ ]:
print("Interactions")

print(
    interactions_clean[
        ["timestamp_utc", "date_utc"]
    ].dtypes
)

print("\nSessions")

print(
    sessions_clean[
        ["start_timestamp_utc", "end_timestamp_utc"]
    ].dtypes
)

## 9. Save Validated Datasets

Save the validated datasets to confirm they are ready for SQL querying and dashboard creation.

In [ ]:
interactions_clean.to_csv("../data/cleaned/llm_system_interactions_cleaned.csv", index=False)

sessions_clean.to_csv("../data/cleaned/llm_system_sessions_summary_cleaned.csv", index=False)

users_clean.to_csv("../data/cleaned/llm_system_users_summary_cleaned.csv", index=False)

prompts_clean.to_csv("../data/cleaned/llm_system_prompts_lookup_cleaned.csv", index=False)

print("Validation completed successfully.")

## Validation Summary

The cleaned datasets were successfully validated.

- Row counts were verified.
- No duplicate records were found.
- Data types are appropriate for analysis.
- Remaining missing values were reviewed and confirmed to represent valid business scenarios (such as missing user feedback, missing CSAT responses, and zero-duration sessions). These values were intentionally retained to preserve data integrity.

The cleaned datasets have been validated successfully and are ready for Power BI dashboard development and further analytical reporting.